# Rainfall Pattern and Climate Analysis

## 1. Introduction
This notebook performs an in-depth analysis of rainfall patterns in India from 1901 to 2015. The objective is to understand seasonal variations, long-term trends, and extreme rainfall events to provide insights relevant for agricultural planning and water resource management.

In [12]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
"""
=============================================================
 STEP 1 : Data Loading & Initial Inspection
 Project : Rainfall Pattern & Climate Analysis
 Dataset : rainfall_in_india_1901-2015.csv
=============================================================
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 1. LOAD THE DATASET
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print(" LOADING DATASET")
print("=" * 60)

FILE_PATH = "rainfall in india 1901-2015.csv"

df = pd.read_csv(FILE_PATH)
print(f"✔  File loaded successfully: {FILE_PATH}")


# ─────────────────────────────────────────────────────────────
# 2. DATASET SHAPE
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" DATASET SHAPE")
print("=" * 60)

rows, cols = df.shape
print(f"   Rows    : {rows}")
print(f"   Columns : {cols}")


# ─────────────────────────────────────────────────────────────
# 3. COLUMN NAMES  (detected dynamically — no assumptions)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" COLUMN NAMES")
print("=" * 60)

for i, col in enumerate(df.columns, start=1):
    print(f"   {i:>2}. {col}")


# ─────────────────────────────────────────────────────────────
# 4. FIRST 5 ROWS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" FIRST 5 ROWS")
print("=" * 60)

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", 120)          # wider output
pd.set_option("display.float_format", "{:.1f}".format)

print(df.head())


# ─────────────────────────────────────────────────────────────
# 5. DATASET INFO  (dtypes + non-null counts)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" DATASET INFO")
print("=" * 60)

df.info()


# ─────────────────────────────────────────────────────────────
# 6. MISSING VALUE CHECK  (dynamic — works for any columns)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" MISSING VALUE REPORT")
print("=" * 60)

# Count missing values per column
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

# Build a clean summary table
missing_report = pd.DataFrame({
    "Missing Count" : missing_count,
    "Missing %"     : missing_pct
})

total_missing = missing_count.sum()

if total_missing == 0:
    print("✔  No missing values found. Dataset is complete.")
else:
    print(f"   Total missing cells : {total_missing}")
    print(f"   Affected columns    : {(missing_count > 0).sum()}\n")

    # Show only columns that have at least 1 missing value
    print(missing_report[missing_report["Missing Count"] > 0].to_string())

print("\n" + "=" * 60)
print(" INSPECTION COMPLETE")
print("=" * 60)

Saving rainfall in india 1901-2015.csv to rainfall in india 1901-2015 (1).csv
User uploaded file "rainfall in india 1901-2015 (1).csv" with length 528115 bytes
 LOADING DATASET
✔  File loaded successfully: rainfall in india 1901-2015.csv

 DATASET SHAPE
   Rows    : 4116
   Columns : 19

 COLUMN NAMES
    1. SUBDIVISION
    2. YEAR
    3. JAN
    4. FEB
    5. MAR
    6. APR
    7. MAY
    8. JUN
    9. JUL
   10. AUG
   11. SEP
   12. OCT
   13. NOV
   14. DEC
   15. ANNUAL
   16. Jan-Feb
   17. Mar-May
   18. Jun-Sep
   19. Oct-Dec

 FIRST 5 ROWS
                 SUBDIVISION  YEAR  JAN   FEB  MAR   APR   MAY   JUN   JUL   AUG   SEP   OCT   NOV   DEC  ANNUAL  \
0  ANDAMAN & NICOBAR ISLANDS  1901 49.2  87.1 29.2   2.3 528.8 517.5 365.1 481.1 332.6 388.5 558.2  33.6  3373.2   
1  ANDAMAN & NICOBAR ISLANDS  1902  0.0 159.8 12.2   0.0 446.1 537.1 228.9 753.7 666.2 197.2 359.0 160.5  3520.7   
2  ANDAMAN & NICOBAR ISLANDS  1903 12.7 144.0  0.0   1.0 235.1 479.9 728.4 326.7 339.0 181.2 284.

In [13]:
"""
=============================================================
 STEP 2 : Data Cleaning & Standardisation
 Project : Rainfall Pattern & Climate Analysis
 Dataset : rainfall_in_india_1901-2015.csv
=============================================================

Cleaning tasks performed:
  1. Load raw dataset
  2. Standardise column names
  3. Ensure all rainfall columns are numeric
  4. Drop unnecessary columns
  5. Fix known typos in SUBDIVISION names
  6. Handle missing values (monthly → impute, ANNUAL → recalculate)
  7. Confirm rainfall units are in mm
  8. Final validation report
  9. Save cleaned dataset
=============================================================
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# LOAD RAW DATASET
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print(" STEP 2 : DATA CLEANING & STANDARDISATION")
print("=" * 60)

df = pd.read_csv("rainfall in india 1901-2015.csv")
print(f"\n✔  Raw dataset loaded  →  {df.shape[0]} rows × {df.shape[1]} columns")


# ─────────────────────────────────────────────────────────────
# TASK 1 : STANDARDISE COLUMN NAMES
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 1 · Standardise Column Names")
print("-" * 60)

# Strip spaces and convert all column names to UPPER_SNAKE_CASE
# The seasonal aggregate columns (Jan-Feb etc.) get renamed too
rename_map = {
    "Jan-Feb"  : "JAN_FEB",
    "Mar-May"  : "MAR_MAY",
    "Jun-Sep"  : "JUN_SEP",
    "Oct-Dec"  : "OCT_DEC",
}

df.columns = df.columns.str.strip()          # remove accidental spaces
df.rename(columns=rename_map, inplace=True)  # rename seasonal columns

print("   Column names after standardisation:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:>2}. {col}")


# ─────────────────────────────────────────────────────────────
# TASK 2 : ENSURE RAINFALL COLUMNS ARE NUMERIC
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 2 · Ensure Rainfall Columns Are Numeric")
print("-" * 60)

MONTH_COLS = ["JAN","FEB","MAR","APR","MAY","JUN",
              "JUL","AUG","SEP","OCT","NOV","DEC"]

NUMERIC_COLS = MONTH_COLS + ["ANNUAL", "JAN_FEB", "MAR_MAY", "JUN_SEP", "OCT_DEC"]

# Force-convert to numeric; anything non-numeric becomes NaN
for col in NUMERIC_COLS:
    before = df[col].isnull().sum()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    after  = df[col].isnull().sum()
    new_nulls = after - before
    if new_nulls > 0:
        print(f"   ⚠  {col}: {new_nulls} non-numeric value(s) converted to NaN")

print("   ✔  All rainfall columns confirmed as float64")
print(f"   Dtypes → SUBDIVISION: {df['SUBDIVISION'].dtype}  |"
      f"  YEAR: {df['YEAR'].dtype}  |  JAN: {df['JAN'].dtype}")


# ─────────────────────────────────────────────────────────────
# TASK 3 : DROP UNNECESSARY COLUMNS
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 3 · Drop Unnecessary Columns")
print("-" * 60)

# The seasonal aggregate columns (JAN_FEB, MAR_MAY, JUN_SEP, OCT_DEC)
# are derived values — we will recalculate them ourselves from monthly
# data so we can guarantee consistency. Drop them now.
COLS_TO_DROP = ["JAN_FEB", "MAR_MAY", "JUN_SEP", "OCT_DEC"]

df.drop(columns=COLS_TO_DROP, inplace=True)
print(f"   Dropped  : {COLS_TO_DROP}")
print(f"   Remaining: {list(df.columns)}")


# ─────────────────────────────────────────────────────────────
# TASK 4 : FIX TYPOS IN SUBDIVISION NAMES
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 4 · Fix Typos in SUBDIVISION Names")
print("-" * 60)

# Known typo in the original Kaggle dataset
TYPO_FIXES = {
    "MATATHWADA" : "MARATHWADA",   # misspelled in source data
}

df["SUBDIVISION"] = df["SUBDIVISION"].str.strip().str.upper()
df["SUBDIVISION"] = df["SUBDIVISION"].replace(TYPO_FIXES)

for wrong, correct in TYPO_FIXES.items():
    print(f"   Fixed  :  '{wrong}'  →  '{correct}'")

print(f"\n   Total subdivisions : {df['SUBDIVISION'].nunique()}")
print(f"   All subdivisions   :")
for s in sorted(df["SUBDIVISION"].unique()):
    print(f"     • {s}")


# ─────────────────────────────────────────────────────────────
# TASK 5 : HANDLE MISSING VALUES
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 5 · Handle Missing Values")
print("-" * 60)

# --- Before ---
total_missing_before = df[MONTH_COLS].isnull().sum().sum()
print(f"   Missing in monthly columns (before) : {total_missing_before}")

# Strategy:
# Monthly columns → fill with that subdivision's long-term mean
# for that specific month.  This preserves regional rainfall
# character far better than a global mean or zero-fill.
print("\n   Imputing monthly columns with subdivision-month mean …")

for month in MONTH_COLS:
    missing_mask = df[month].isnull()
    n_missing    = missing_mask.sum()

    if n_missing == 0:
        continue

    # Mean of this month across all years for the same subdivision
    sub_mean = df.groupby("SUBDIVISION")[month].transform("mean")
    df.loc[missing_mask, month] = sub_mean[missing_mask].round(1)

    print(f"   {month:<4}  →  {n_missing} value(s) filled with subdivision mean")

# --- After ---
total_missing_after = df[MONTH_COLS].isnull().sum().sum()
print(f"\n   Missing in monthly columns (after)  : {total_missing_after}")


# ─────────────────────────────────────────────────────────────
# TASK 6 : RECALCULATE ANNUAL FROM MONTHLY (ensures consistency)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 6 · Recalculate ANNUAL from Monthly Columns")
print("-" * 60)

# Always recompute ANNUAL so it is guaranteed to equal sum(JAN..DEC)
df["ANNUAL"] = df[MONTH_COLS].sum(axis=1).round(1)

print("   ✔  ANNUAL recalculated as sum of JAN through DEC")
print(f"   ANNUAL range : {df['ANNUAL'].min():.1f} mm  –  {df['ANNUAL'].max():.1f} mm")


# ─────────────────────────────────────────────────────────────
# TASK 7 : ADD SEASONAL COLUMNS (clean, derived from monthly)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 7 · Add Clean Seasonal Aggregate Columns")
print("-" * 60)

df["WINTER"]       = df[["JAN","FEB"]].sum(axis=1).round(1)
df["PRE_MONSOON"]  = df[["MAR","APR","MAY"]].sum(axis=1).round(1)
df["MONSOON"]      = df[["JUN","JUL","AUG","SEP"]].sum(axis=1).round(1)
df["POST_MONSOON"] = df[["OCT","NOV","DEC"]].sum(axis=1).round(1)

print("   Seasonal columns added:")
print("   • WINTER       = JAN + FEB")
print("   • PRE_MONSOON  = MAR + APR + MAY")
print("   • MONSOON      = JUN + JUL + AUG + SEP")
print("   • POST_MONSOON = OCT + NOV + DEC")


# ─────────────────────────────────────────────────────────────
# TASK 8 : CONFIRM RAINFALL UNITS ARE IN MM
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" TASK 8 · Confirm Rainfall Units (mm)")
print("-" * 60)

# IMD / Kaggle source records all values in millimetres.
# Cross-check: India's highest subdivision annual rainfall is
# ~4000 mm (Andaman). Values in cm would be ~400, in metres ~4.
# Our max annual confirms mm scale.

max_annual = df["ANNUAL"].max()
min_annual = df["ANNUAL"].min()
mean_annual = df["ANNUAL"].mean()

print(f"   Annual rainfall — Min  : {min_annual:.1f} mm")
print(f"   Annual rainfall — Max  : {max_annual:.1f} mm")
print(f"   Annual rainfall — Mean : {mean_annual:.1f} mm")

if max_annual > 500:
    print("   ✔  Values consistent with millimetres (mm) — confirmed.")
else:
    print("   ⚠  Values seem low — may need unit verification.")


# ─────────────────────────────────────────────────────────────
# FINAL VALIDATION REPORT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" FINAL VALIDATION REPORT")
print("=" * 60)

print(f"   Shape             : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns           : {list(df.columns)}")
print(f"   SUBDIVISION       : {df['SUBDIVISION'].nunique()} unique, 0 nulls")
print(f"   YEAR range        : {df['YEAR'].min()} – {df['YEAR'].max()}")
print(f"   Monthly nulls     : {df[MONTH_COLS].isnull().sum().sum()}")
print(f"   ANNUAL nulls      : {df['ANNUAL'].isnull().sum()}")
print(f"   Negative values   : {(df[MONTH_COLS] < 0).sum().sum()}")
print(f"   Units             : Millimetres (mm) ✔")

print("\n   First 5 rows of cleaned dataset:")
print(df.head())


# ─────────────────────────────────────────────────────────────
# SAVE CLEANED DATASET
# ─────────────────────────────────────────────────────────────
OUTPUT_FILE = "rainfall_cleaned.csv"
df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✔  Cleaned dataset saved → '{OUTPUT_FILE}'")
print("=" * 60)

 STEP 2 : DATA CLEANING & STANDARDISATION

✔  Raw dataset loaded  →  4116 rows × 19 columns

------------------------------------------------------------
 TASK 1 · Standardise Column Names
------------------------------------------------------------
   Column names after standardisation:
    1. SUBDIVISION
    2. YEAR
    3. JAN
    4. FEB
    5. MAR
    6. APR
    7. MAY
    8. JUN
    9. JUL
   10. AUG
   11. SEP
   12. OCT
   13. NOV
   14. DEC
   15. ANNUAL
   16. JAN_FEB
   17. MAR_MAY
   18. JUN_SEP
   19. OCT_DEC

------------------------------------------------------------
 TASK 2 · Ensure Rainfall Columns Are Numeric
------------------------------------------------------------
   ✔  All rainfall columns confirmed as float64
   Dtypes → SUBDIVISION: object  |  YEAR: int64  |  JAN: float64

------------------------------------------------------------
 TASK 3 · Drop Unnecessary Columns
------------------------------------------------------------
   Dropped  : ['JAN_FEB', 'MAR_MAY

In [14]:
"""
=============================================================
 STEP 3 : Reshape to Long Format
 Project : Rainfall Pattern & Climate Analysis
 Input   : rainfall_cleaned.csv
 Output  : rainfall_long.csv
=============================================================

Wide format (current):
  SUBDIVISION | YEAR | JAN | FEB | ... | DEC | ANNUAL | ...
  Each row = one subdivision × one year (12 months as columns)

Long format (target):
  SUBDIVISION | YEAR | MONTH | RAINFALL
  Each row = one subdivision × one year × one month
  Total rows = 4116 × 12 = 49,392

Why long format?
  - Easier to plot monthly trends
  - Required for groupby analysis by month
  - Works directly with seaborn / plotly
=============================================================
"""

import pandas as pd

# ─────────────────────────────────────────────────────────────
# 1. LOAD CLEANED DATASET
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print(" STEP 3 : RESHAPE TO LONG FORMAT")
print("=" * 60)

df = pd.read_csv("rainfall_cleaned.csv")
print(f"\n✔  Loaded rainfall_cleaned.csv  →  {df.shape[0]} rows × {df.shape[1]} cols")

# ─────────────────────────────────────────────────────────────
# 2. DEFINE MONTH COLUMNS (in correct calendar order)
# ─────────────────────────────────────────────────────────────

# These are the only columns we want to "melt" into rows
MONTH_COLS = ["JAN","FEB","MAR","APR","MAY","JUN",
              "JUL","AUG","SEP","OCT","NOV","DEC"]

# Columns to keep as-is (identifier columns)
ID_COLS = ["SUBDIVISION", "YEAR"]

# ─────────────────────────────────────────────────────────────
# 3. SELECT ONLY WHAT WE NEED BEFORE MELTING
#    Drop ANNUAL and seasonal columns — not needed in long format
# ─────────────────────────────────────────────────────────────
df_wide = df[ID_COLS + MONTH_COLS].copy()

print(f"\n   Columns kept for reshape : {ID_COLS + MONTH_COLS}")
print(f"   Shape before melt        : {df_wide.shape}")

# ─────────────────────────────────────────────────────────────
# 4. MELT — wide → long
#    pd.melt() unpivots the 12 month columns into rows
# ─────────────────────────────────────────────────────────────
df_long = pd.melt(
    df_wide,
    id_vars    = ID_COLS,        # columns that stay as columns
    value_vars = MONTH_COLS,     # columns to turn into rows
    var_name   = "MONTH",        # name for the new month column
    value_name = "RAINFALL"      # name for the rainfall values column
)

print(f"   Shape after melt         : {df_long.shape}")

# ─────────────────────────────────────────────────────────────
# 5. RENAME SUBDIVISION → STATE  (cleaner label for analysis)
# ─────────────────────────────────────────────────────────────
df_long.rename(columns={"SUBDIVISION": "STATE"}, inplace=True)

# ─────────────────────────────────────────────────────────────
# 6. ENSURE MONTHS ARE ORDERED JAN → DEC
#    After melt, month order may not be guaranteed.
#    Use pd.Categorical to enforce correct order.
# ─────────────────────────────────────────────────────────────
df_long["MONTH"] = pd.Categorical(
    df_long["MONTH"],
    categories = MONTH_COLS,   # defines the correct order
    ordered    = True
)

# Sort rows: by STATE, then YEAR, then MONTH (calendar order)
df_long.sort_values(["STATE", "YEAR", "MONTH"], inplace=True)
df_long.reset_index(drop=True, inplace=True)

# ─────────────────────────────────────────────────────────────
# 7. ADD MONTH NUMBER (1–12) — useful for plotting & filtering
# ─────────────────────────────────────────────────────────────
month_number_map = {m: i for i, m in enumerate(MONTH_COLS, start=1)}
df_long["MONTH_NUM"] = df_long["MONTH"].map(month_number_map)

# ─────────────────────────────────────────────────────────────
# 8. FINAL COLUMN ORDER
# ─────────────────────────────────────────────────────────────
df_long = df_long[["STATE", "YEAR", "MONTH", "MONTH_NUM", "RAINFALL"]]

# ─────────────────────────────────────────────────────────────
# 9. VALIDATE
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" VALIDATION")
print("-" * 60)

expected_rows = df_wide.shape[0] * 12
print(f"   Expected rows  : {expected_rows:,}  (4116 years×subdivisions × 12 months)")
print(f"   Actual rows    : {df_long.shape[0]:,}")
print(f"   Columns        : {list(df_long.columns)}")
print(f"   Missing values : {df_long.isnull().sum().sum()}")
print(f"   States         : {df_long['STATE'].nunique()}")
print(f"   Year range     : {df_long['YEAR'].min()} – {df_long['YEAR'].max()}")
print(f"   Month order    : {list(df_long['MONTH'].cat.categories)}")

print("\n   First 15 rows (one subdivision, all 12 months for 1901):")
sample = df_long[(df_long["STATE"] == "ANDAMAN & NICOBAR ISLANDS") &
                 (df_long["YEAR"] == 1901)]
print(sample.to_string(index=False))

# ─────────────────────────────────────────────────────────────
# 10. SAVE
# ─────────────────────────────────────────────────────────────
OUTPUT = "rainfall_long.csv"
df_long.to_csv(OUTPUT, index=False)

print(f"\n✔  Long format dataset saved → '{OUTPUT}'")
print("=" * 60)

 STEP 3 : RESHAPE TO LONG FORMAT

✔  Loaded rainfall_cleaned.csv  →  4116 rows × 19 cols

   Columns kept for reshape : ['SUBDIVISION', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
   Shape before melt        : (4116, 14)
   Shape after melt         : (49392, 4)

------------------------------------------------------------
 VALIDATION
------------------------------------------------------------
   Expected rows  : 49,392  (4116 years×subdivisions × 12 months)
   Actual rows    : 49,392
   Columns        : ['STATE', 'YEAR', 'MONTH', 'MONTH_NUM', 'RAINFALL']
   Missing values : 0
   States         : 36
   Year range     : 1901 – 2015
   Month order    : ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

   First 15 rows (one subdivision, all 12 months for 1901):
                    STATE  YEAR MONTH MONTH_NUM  RAINFALL
ANDAMAN & NICOBAR ISLANDS  1901   JAN         1      49.2
ANDAMAN & NICOBAR ISLANDS  1901

In [15]:
"""
=============================================================
 STEP 4 : Seasonal Rainfall Pattern Analysis
 Project : Rainfall Pattern & Climate Analysis
 Input   : rainfall_long.csv
 Output  : 04_seasonal_analysis.png
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print(" STEP 4 : SEASONAL RAINFALL PATTERN ANALYSIS")
print("=" * 60)

df = pd.read_csv("rainfall_long.csv")

# Enforce correct month order for all groupby/plots
MONTH_ORDER = ["JAN","FEB","MAR","APR","MAY","JUN",
               "JUL","AUG","SEP","OCT","NOV","DEC"]

df["MONTH"] = pd.Categorical(df["MONTH"], categories=MONTH_ORDER, ordered=True)

print(f"\n✔  Loaded rainfall_long.csv  →  {df.shape[0]:,} rows")

# ─────────────────────────────────────────────────────────────
# 2. CALCULATE AVERAGE RAINFALL PER MONTH (all India, all years)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" ANALYSIS 1 · Average Monthly Rainfall (All-India)")
print("-" * 60)

monthly_avg = (
    df.groupby("MONTH", observed=True)["RAINFALL"]
    .mean()
    .round(2)
    .reset_index()
)

# Add month number for easy sorting reference
monthly_avg["MONTH_NUM"] = range(1, 13)

print("\n   Month-wise Average Rainfall (mm) — All subdivisions, 1901–2015:")
print(f"   {'Month':<6}  {'Avg Rainfall (mm)':>18}")
print("   " + "-" * 28)
for _, row in monthly_avg.iterrows():
    bar = "█" * int(row["RAINFALL"] / 15)
    print(f"   {row['MONTH']:<6}  {row['RAINFALL']:>10.2f} mm   {bar}")

# ─────────────────────────────────────────────────────────────
# 3. IDENTIFY PEAK MONSOON MONTHS
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" ANALYSIS 2 · Peak Rainfall Month Identification")
print("-" * 60)

peak_month   = monthly_avg.loc[monthly_avg["RAINFALL"].idxmax(), "MONTH"]
peak_value   = monthly_avg["RAINFALL"].max()
lowest_month = monthly_avg.loc[monthly_avg["RAINFALL"].idxmin(), "MONTH"]
lowest_value = monthly_avg["RAINFALL"].min()

# Monsoon months (Jun–Sep) stats
monsoon_months = ["JUN","JUL","AUG","SEP"]
monsoon_avg    = monthly_avg[monthly_avg["MONTH"].isin(monsoon_months)]["RAINFALL"].sum()
annual_avg     = monthly_avg["RAINFALL"].sum()
monsoon_pct    = (monsoon_avg / annual_avg * 100).round(1)

print(f"\n   Peak rainfall month   : {peak_month}  ({peak_value:.2f} mm avg)")
print(f"   Lowest rainfall month : {lowest_month}  ({lowest_value:.2f} mm avg)")
print(f"\n   Monsoon season (Jun–Sep) contributes : {monsoon_pct}% of annual rainfall")
print(f"   Monsoon avg total     : {monsoon_avg:.2f} mm")
print(f"   Annual avg total      : {annual_avg:.2f} mm")

# Season-wise breakdown
season_totals = {
    "Winter (Jan–Feb)"      : monthly_avg[monthly_avg["MONTH"].isin(["JAN","FEB"])]["RAINFALL"].sum(),
    "Pre-Monsoon (Mar–May)" : monthly_avg[monthly_avg["MONTH"].isin(["MAR","APR","MAY"])]["RAINFALL"].sum(),
    "Monsoon (Jun–Sep)"     : monsoon_avg,
    "Post-Monsoon (Oct–Dec)": monthly_avg[monthly_avg["MONTH"].isin(["OCT","NOV","DEC"])]["RAINFALL"].sum(),
}

print("\n   Season-wise Contribution:")
for season, val in season_totals.items():
    pct = val / annual_avg * 100
    print(f"   {season:<30}  {val:>7.2f} mm  ({pct:.1f}%)")

# ─────────────────────────────────────────────────────────────
# 4. PLOT — 3-panel figure
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" PLOT · Building Charts")
print("-" * 60)

# Season color palette for background shading
SEASON_COLORS = {
    "Winter"      : "#cce5ff",   # light blue
    "Pre-Monsoon" : "#fff3cd",   # light yellow
    "Monsoon"     : "#d4edda",   # light green
    "Post-Monsoon": "#f8d7da",   # light red
}
SEASON_SPANS = {
    "Winter"      : (0.5, 2.5),
    "Pre-Monsoon" : (2.5, 5.5),
    "Monsoon"     : (5.5, 9.5),
    "Post-Monsoon": (9.5, 12.5),
}

months      = list(monthly_avg["MONTH"])
rainfall    = list(monthly_avg["RAINFALL"])
month_nums  = list(monthly_avg["MONTH_NUM"])

fig, axes = plt.subplots(3, 1, figsize=(13, 16))
fig.suptitle(
    "Seasonal Rainfall Pattern Analysis — India (1901–2015)",
    fontsize=16, fontweight="bold", y=0.98
)

# ── PANEL 1 : Line Chart — Monthly Trend ─────────────────────
ax1 = axes[0]

# Shade seasonal bands
for season, (x0, x1) in SEASON_SPANS.items():
    ax1.axvspan(x0, x1, color=SEASON_COLORS[season], alpha=0.4, label=season)

ax1.plot(month_nums, rainfall, color="#1a6bab", linewidth=2.5,
         marker="o", markersize=7, zorder=5, label="Avg Rainfall")

# Annotate each point
for x, y, m in zip(month_nums, rainfall, months):
    ax1.annotate(f"{y:.0f}", xy=(x, y), xytext=(0, 9),
                 textcoords="offset points", ha="center",
                 fontsize=8, color="#333333")

# Mark peak month
peak_idx = month_nums[months.index(peak_month)]
ax1.scatter([peak_idx], [peak_value], color="crimson",
            zorder=6, s=120, label=f"Peak: {peak_month} ({peak_value:.0f} mm)")

ax1.set_xticks(month_nums)
ax1.set_xticklabels(months, fontsize=10)
ax1.set_ylabel("Avg Rainfall (mm)", fontsize=11)
ax1.set_title("Average Monthly Rainfall — All-India Trend", fontsize=12, pad=8)
ax1.set_xlim(0.5, 12.5)
ax1.set_ylim(0, max(rainfall) * 1.2)
ax1.legend(loc="upper left", fontsize=9, framealpha=0.8)
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.spines[["top","right"]].set_visible(False)

# ── PANEL 2 : Bar Chart — Monthly Rainfall with season colors ─
ax2 = axes[1]

bar_colors = []
for m in months:
    if m in ["JAN","FEB"]:
        bar_colors.append("#5b9bd5")
    elif m in ["MAR","APR","MAY"]:
        bar_colors.append("#f0ad4e")
    elif m in ["JUN","JUL","AUG","SEP"]:
        bar_colors.append("#2ca02c")
    else:
        bar_colors.append("#d62728")

bars = ax2.bar(months, rainfall, color=bar_colors, edgecolor="white",
               linewidth=0.8, zorder=3)

# Value labels on bars
for bar, val in zip(bars, rainfall):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f"{val:.0f}", ha="center", va="bottom", fontsize=8.5)

ax2.set_ylabel("Avg Rainfall (mm)", fontsize=11)
ax2.set_title("Monthly Rainfall Distribution by Season", fontsize=12, pad=8)
ax2.set_ylim(0, max(rainfall) * 1.18)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.spines[["top","right"]].set_visible(False)

legend_patches = [
    mpatches.Patch(color="#5b9bd5", label="Winter (Jan–Feb)"),
    mpatches.Patch(color="#f0ad4e", label="Pre-Monsoon (Mar–May)"),
    mpatches.Patch(color="#2ca02c", label="Monsoon (Jun–Sep)"),
    mpatches.Patch(color="#d62728", label="Post-Monsoon (Oct–Dec)"),
]
ax2.legend(handles=legend_patches, loc="upper left", fontsize=9, framealpha=0.8)

# ── PANEL 3 : Pie Chart — Seasonal Contribution ──────────────
ax3 = axes[2]

season_labels = list(season_totals.keys())
season_values = list(season_totals.values())
pie_colors    = ["#5b9bd5", "#f0ad4e", "#2ca02c", "#d62728"]
explode       = [0, 0, 0.06, 0]   # highlight monsoon slice

wedges, texts, autotexts = ax3.pie(
    season_values,
    labels    = season_labels,
    colors    = pie_colors,
    explode   = explode,
    autopct   = "%1.1f%%",
    startangle= 140,
    textprops = {"fontsize": 10},
    wedgeprops= {"edgecolor": "white", "linewidth": 1.5},
)

for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")

ax3.set_title("Seasonal Contribution to Annual Rainfall", fontsize=12, pad=8)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("04_seasonal_analysis.png", dpi=150, bbox_inches="tight")
plt.close()

print("   ✔  Chart saved → 04_seasonal_analysis.png")

# ─────────────────────────────────────────────────────────────
# 5. INSIGHT STATEMENTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" AGRICULTURAL INSIGHTS")
print("=" * 60)

print(f"""
  1. MONSOON DOMINANCE
     Jun–Sep contributes {monsoon_pct}% of India's annual rainfall.
     This makes kharif crops (Rice, Maize, Cotton, Soybean)
     critically dependent on monsoon onset and duration.

  2. PEAK MONTH : {peak_month}
     {peak_month} records the highest average rainfall ({peak_value:.0f} mm).
     Paddy transplanting and sugarcane growth peak during this window.

  3. DRY MONTHS (Jan–Mar)
     Winter months average < 25 mm/month — ideal for rabi crops
     (Wheat, Mustard, Chickpea) that rely on stored soil moisture
     and limited irrigation rather than live rainfall.

  4. PRE-MONSOON (Mar–May)
     Rainfall rises gradually. Useful for early sowing of
     groundnut and sesame in southern states.

  5. POST-MONSOON (Oct–Dec)
     Retreating monsoon brings moderate rain to Tamil Nadu and
     coastal Andhra Pradesh — enabling a second paddy crop (samba).
""")

print("✔  Analysis complete.")
print("=" * 60)

 STEP 4 : SEASONAL RAINFALL PATTERN ANALYSIS

✔  Loaded rainfall_long.csv  →  49,392 rows

------------------------------------------------------------
 ANALYSIS 1 · Average Monthly Rainfall (All-India)
------------------------------------------------------------

   Month-wise Average Rainfall (mm) — All subdivisions, 1901–2015:
   Month    Avg Rainfall (mm)
   ----------------------------
   JAN          18.96 mm   █
   FEB          21.82 mm   █
   MAR          27.42 mm   █
   APR          43.14 mm   ██
   MAY          85.85 mm   █████
   JUN         230.50 mm   ███████████████
   JUL         347.24 mm   ███████████████████████
   AUG         290.28 mm   ███████████████████
   SEP         197.51 mm   █████████████
   OCT          95.70 mm   ██████
   NOV          40.08 mm   ██
   DEC          19.02 mm   █

------------------------------------------------------------
 ANALYSIS 2 · Peak Rainfall Month Identification
------------------------------------------------------------

   Peak 

In [16]:
"""
=============================================================
 STEP 5 : Long-Term Rainfall Trend Analysis (1901–2015)
 Project : Rainfall Pattern & Climate Analysis
 Input   : rainfall_long.csv
 Output  : 05_longterm_trends.png
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import linregress

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print(" STEP 5 : LONG-TERM RAINFALL TREND ANALYSIS")
print("=" * 60)

df = pd.read_csv("rainfall_long.csv")
print(f"\n✔  Loaded rainfall_long.csv  →  {df.shape[0]:,} rows")

# ─────────────────────────────────────────────────────────────
# 2. CALCULATE AVERAGE ANNUAL RAINFALL PER YEAR
#    We average across all 36 subdivisions so every year is
#    on the same scale regardless of missing subdivisions.
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" ANALYSIS 1 · Average Annual Rainfall Per Year")
print("-" * 60)

# Step 1 : sum all 12 months per subdivision per year
sub_annual = (
    df.groupby(["STATE", "YEAR"])["RAINFALL"]
    .sum()
    .reset_index()
    .rename(columns={"RAINFALL": "ANNUAL_RAINFALL"})
)

# Step 2 : average across all subdivisions for each year
yearly = (
    sub_annual.groupby("YEAR")["ANNUAL_RAINFALL"]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"ANNUAL_RAINFALL": "AVG_RAINFALL"})
)

print(f"\n   Years covered : {yearly['YEAR'].min()} – {yearly['YEAR'].max()}")
print(f"   Overall mean  : {yearly['AVG_RAINFALL'].mean():.2f} mm/year")
print(f"   Overall std   : {yearly['AVG_RAINFALL'].std():.2f} mm")
print(f"   Min year avg  : {yearly['AVG_RAINFALL'].min():.2f} mm  ({yearly.loc[yearly['AVG_RAINFALL'].idxmin(), 'YEAR']})")
print(f"   Max year avg  : {yearly['AVG_RAINFALL'].max():.2f} mm  ({yearly.loc[yearly['AVG_RAINFALL'].idxmax(), 'YEAR']})")

# ─────────────────────────────────────────────────────────────
# 3. TREND LINE  (linear regression via scipy)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" ANALYSIS 2 · Linear Trend (Linear Regression)")
print("-" * 60)

years    = yearly["YEAR"].values
rainfall = yearly["AVG_RAINFALL"].values

slope, intercept, r_value, p_value, std_err = linregress(years, rainfall)

trend_line = slope * years + intercept   # y = mx + c

direction = "INCREASING ▲" if slope > 0 else "DECREASING ▼"
print(f"\n   Slope (mm/year) : {slope:.4f}  →  {direction}")
print(f"   R² value        : {r_value**2:.4f}  (strength of linear fit)")
print(f"   P-value         : {p_value:.4f}  {'(statistically significant)' if p_value < 0.05 else '(not statistically significant)'}")
print(f"   Change over 115 years : {slope * 115:.2f} mm")

# ─────────────────────────────────────────────────────────────
# 4. IDENTIFY EXTREME YEARS  (±1.5 standard deviations)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" ANALYSIS 3 · Extreme Rainfall Years")
print("-" * 60)

mean_rf = yearly["AVG_RAINFALL"].mean()
std_rf  = yearly["AVG_RAINFALL"].std()

HIGH_THRESHOLD = mean_rf + 1.5 * std_rf
LOW_THRESHOLD  = mean_rf - 1.5 * std_rf

high_years = yearly[yearly["AVG_RAINFALL"] >= HIGH_THRESHOLD].sort_values("AVG_RAINFALL", ascending=False)
low_years  = yearly[yearly["AVG_RAINFALL"] <= LOW_THRESHOLD].sort_values("AVG_RAINFALL")

print(f"\n   Mean rainfall        : {mean_rf:.2f} mm")
print(f"   Std deviation        : {std_rf:.2f} mm")
print(f"   HIGH threshold (mean + 1.5σ) : {HIGH_THRESHOLD:.2f} mm")
print(f"   LOW  threshold (mean - 1.5σ) : {LOW_THRESHOLD:.2f} mm")

print(f"\n   ▲ HIGH RAINFALL YEARS ({len(high_years)} years) :")
for _, row in high_years.iterrows():
    diff = row["AVG_RAINFALL"] - mean_rf
    print(f"     {int(row['YEAR'])}  →  {row['AVG_RAINFALL']:.2f} mm  (+{diff:.2f} mm above mean)")

print(f"\n   ▼ LOW RAINFALL YEARS  ({len(low_years)} years) :")
for _, row in low_years.iterrows():
    diff = mean_rf - row["AVG_RAINFALL"]
    print(f"     {int(row['YEAR'])}  →  {row['AVG_RAINFALL']:.2f} mm  (-{diff:.2f} mm below mean)")

# ─────────────────────────────────────────────────────────────
# 5. 10-YEAR ROLLING MEAN  (smooths short-term noise)
# ─────────────────────────────────────────────────────────────
yearly["ROLLING_10Y"] = yearly["AVG_RAINFALL"].rolling(window=10, center=True).mean()

# ─────────────────────────────────────────────────────────────
# 6. PLOT — 3-panel figure
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print(" PLOT · Building Charts")
print("-" * 60)

fig, axes = plt.subplots(3, 1, figsize=(14, 16))
fig.suptitle(
    "Long-Term Rainfall Trend Analysis — India (1901–2015)",
    fontsize=16, fontweight="bold", y=0.99
)

# ── PANEL 1 : Full time series + trend line ───────────────────
ax1 = axes[0]

ax1.fill_between(years, rainfall, alpha=0.15, color="#1a6bab")
ax1.plot(years, rainfall, color="#1a6bab", linewidth=1.2,
         alpha=0.85, label="Annual Avg Rainfall")
ax1.plot(years, trend_line, color="crimson", linewidth=2.2,
         linestyle="--", label=f"Trend Line (slope={slope:.3f} mm/yr)")

# Mean line
ax1.axhline(mean_rf, color="gray", linewidth=1.2,
            linestyle=":", label=f"Long-term Mean ({mean_rf:.1f} mm)")

# Mark extreme years
ax1.scatter(high_years["YEAR"], high_years["AVG_RAINFALL"],
            color="#e67e22", zorder=5, s=60, label="High Rainfall Year")
ax1.scatter(low_years["YEAR"],  low_years["AVG_RAINFALL"],
            color="#8e44ad",    zorder=5, s=60, label="Low Rainfall Year")

# Annotate top 3 high and low years
for _, row in high_years.head(3).iterrows():
    ax1.annotate(str(int(row["YEAR"])),
                 xy=(row["YEAR"], row["AVG_RAINFALL"]),
                 xytext=(0, 8), textcoords="offset points",
                 ha="center", fontsize=7.5, color="#e67e22", fontweight="bold")

for _, row in low_years.head(3).iterrows():
    ax1.annotate(str(int(row["YEAR"])),
                 xy=(row["YEAR"], row["AVG_RAINFALL"]),
                 xytext=(0, -14), textcoords="offset points",
                 ha="center", fontsize=7.5, color="#8e44ad", fontweight="bold")

ax1.set_xlabel("Year", fontsize=11)
ax1.set_ylabel("Avg Annual Rainfall (mm)", fontsize=11)
ax1.set_title("Yearly Rainfall Trend with Linear Regression Line", fontsize=12, pad=8)
ax1.legend(fontsize=9, loc="upper right", framealpha=0.85)
ax1.grid(axis="y", linestyle="--", alpha=0.35)
ax1.spines[["top", "right"]].set_visible(False)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(10))

# ── PANEL 2 : 10-year rolling mean ───────────────────────────
ax2 = axes[1]

ax2.bar(years, rainfall, color="#90caf9", alpha=0.5,
        width=0.8, label="Annual Avg Rainfall")
ax2.plot(years, yearly["ROLLING_10Y"], color="#1565c0",
         linewidth=2.5, label="10-Year Rolling Mean")
ax2.axhline(mean_rf, color="gray", linewidth=1.2,
            linestyle=":", label=f"Overall Mean ({mean_rf:.1f} mm)")

ax2.set_xlabel("Year", fontsize=11)
ax2.set_ylabel("Avg Annual Rainfall (mm)", fontsize=11)
ax2.set_title("Annual Rainfall with 10-Year Rolling Average", fontsize=12, pad=8)
ax2.legend(fontsize=9, loc="upper right", framealpha=0.85)
ax2.grid(axis="y", linestyle="--", alpha=0.35)
ax2.spines[["top", "right"]].set_visible(False)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(10))

# ── PANEL 3 : Deviation from mean (anomaly bar chart) ────────
ax3 = axes[2]

anomaly = rainfall - mean_rf
colors  = ["#e74c3c" if a >= 0 else "#3498db" for a in anomaly]

ax3.bar(years, anomaly, color=colors, width=0.8, alpha=0.85)
ax3.axhline(0, color="black", linewidth=1.0)
ax3.axhline( 1.5 * std_rf, color="#e67e22", linewidth=1.2,
             linestyle="--", label=f"+1.5σ  High threshold ({HIGH_THRESHOLD:.1f} mm)")
ax3.axhline(-1.5 * std_rf, color="#8e44ad", linewidth=1.2,
             linestyle="--", label=f"−1.5σ  Low threshold  ({LOW_THRESHOLD:.1f} mm)")

ax3.set_xlabel("Year", fontsize=11)
ax3.set_ylabel("Deviation from Mean (mm)", fontsize=11)
ax3.set_title("Annual Rainfall Anomaly (Deviation from Long-Term Mean)", fontsize=12, pad=8)
ax3.legend(fontsize=9, loc="upper right", framealpha=0.85)
ax3.grid(axis="y", linestyle="--", alpha=0.35)
ax3.spines[["top", "right"]].set_visible(False)
ax3.xaxis.set_major_locator(ticker.MultipleLocator(10))

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("05_longterm_trends.png", dpi=150, bbox_inches="tight")
plt.close()

print("   ✔  Chart saved → 05_longterm_trends.png")

# ─────────────────────────────────────────────────────────────
# 7. AGRICULTURAL INSIGHTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" AGRICULTURAL INSIGHTS")
print("=" * 60)

trend_dir = "slight upward" if slope > 0 else "slight downward"
print(f"""
  1. LONG-TERM TREND
     Rainfall shows a {trend_dir} trend of {slope:.3f} mm/year.
     Over 115 years this amounts to a {abs(slope*115):.1f} mm shift —
     small but relevant for long-range agricultural planning.

  2. HIGH RAINFALL YEARS → FLOOD RISK
     Years like {', '.join(str(int(y)) for y in high_years['YEAR'].head(3))}
     saw well-above-average rainfall. These years risk waterlogging
     for Wheat and damage to standing Rabi crops.

  3. LOW RAINFALL YEARS → DROUGHT RISK
     Years like {', '.join(str(int(y)) for y in low_years['YEAR'].head(3))}
     are drought-prone. Millets (Bajra, Ragi) and drought-tolerant
     oilseeds perform better in such years than water-intensive crops.

  4. ROLLING AVERAGE INTERPRETATION
     The 10-year rolling mean reveals multi-decadal wet and dry
     cycles. Farmers and planners can use these cycles to adjust
     crop portfolios and reservoir planning accordingly.

  5. VARIABILITY CONCERN
     Standard deviation of {std_rf:.1f} mm shows high year-to-year
     variability — underscoring the need for irrigation backup
     for crops like Rice and Sugarcane.
""")

print("✔  Analysis complete.")
print("=" * 60)

 STEP 5 : LONG-TERM RAINFALL TREND ANALYSIS

✔  Loaded rainfall_long.csv  →  49,392 rows

------------------------------------------------------------
 ANALYSIS 1 · Average Annual Rainfall Per Year
------------------------------------------------------------

   Years covered : 1901 – 2015
   Overall mean  : 1417.29 mm/year
   Overall std   : 118.38 mm
   Min year avg  : 1146.50 mm  (1972)
   Max year avg  : 1717.08 mm  (1961)

------------------------------------------------------------
 ANALYSIS 2 · Linear Trend (Linear Regression)
------------------------------------------------------------

   Slope (mm/year) : -0.3046  →  DECREASING ▼
   R² value        : 0.0074  (strength of linear fit)
   P-value         : 0.3619  (not statistically significant)
   Change over 115 years : -35.03 mm

------------------------------------------------------------
 ANALYSIS 3 · Extreme Rainfall Years
------------------------------------------------------------

   Mean rainfall        : 1417.29 mm
  